In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import approx_fprime

# =====================================================================
# 1. SETUP DATA AND PARAMETERS (MATCHING PART 3)
# =====================================================================
# Feature Matrix X (2 data points, 2 features each)
X = np.array([[1.0, 3.0],
              [4.0, 10.0]])

# True Target Vector y
y = np.array([5.0, 6.0])

# Initial parameters (Must match Part 3 precisely)
m_initial = np.array([-1.0, 2.0])
b_initial = np.array([1.0, 1.0])

learning_rate = 0.01
num_iterations = 20  # Expanded slightly to show a beautiful convergence trend

# =====================================================================
# 2. DRY PRINCIPLE: MODULAR FUNCTIONS
# =====================================================================
def compute_predictions(X, m, b):
    """Computes y_hat = Xm^T + b via matrix multiplication."""
    return np.dot(X, m) + b

def compute_mse(m_flat, X, y, b):
    """
    Cost function structured specifically for SciPy's gradient checker.
    SciPy expects the optimized parameters to be a flat array.
    """
    preds = compute_predictions(X, m_flat, b)
    error = preds - y
    return np.mean(error ** 2)

# =====================================================================
# 3. NON-ABSTRACTED GRADIENT DESCENT LOOP
# =====================================================================
# Arrays to track parameters and error for plotting
history_m1 = []
history_m2 = []
history_b = []
history_loss = []

# Copy initial weights to mutable tracking variables
m = m_initial.copy()
b = b_initial.copy()

print("====================================================================")
print("             GROUP 12: GRADIENT DESCENT VERIFICATION               ")
print("====================================================================\n")

for i in range(num_iterations):
    # Step A: Compute Predictions (Forward Pass)
    y_hat = compute_predictions(X, m, b)

    # Step B: Calculate the explicit Error Vector
    error_vector = y_hat - y

    # Step C: Explicit Matrix Gradient Calculation (Visible & Step-by-Step)
    N = len(y)
    grad_m = (2.0 / N) * np.dot(error_vector, X)
    # Since b is a vector [1,1], its gradient shares the scalar sum of errors
    grad_b_scalar = (2.0 / N) * np.sum(error_vector)
    grad_b = np.array([grad_b_scalar, grad_b_scalar])

    # Step D: SciPy Numerical Gradient Verification
    # approx_fprime uses finite differences to verify our manual calculus derivatives
    scipy_grad_m = approx_fprime(m, compute_mse, 1e-6, X, y, b)

    # Track metrics for the first 4 iterations to cross-examine with Part 3
    if i < 4:
        loss = np.mean(error_vector ** 2)
        print(f"--- Iteration {i+1} (Calculated Step) ---")
        print(f"  Predictions (y_hat) : {y_hat}")
        print(f"  Error Vector (e)    : {error_vector}")
        print(f"  Manual Gradient m   : {grad_m}")
        print(f"  SciPy Verified Grad : {scipy_grad_m}")
        print(f"  Updated Parameters  : m = {m - learning_rate * grad_m}, b = {b - learning_rate * grad_b}")
        print(f"  Current MSE Loss    : {loss:.4f}\n")

    # Record history
    history_m1.append(m[0])
    history_m2.append(m[1])
    history_b.append(b[0])
    history_loss.append(np.mean(error_vector ** 2))

    # Step E: Apply Parameter Update Rule
    m = m - learning_rate * grad_m
    b = b - learning_rate * grad_b

# Final Model Synthesis
final_predictions = compute_predictions(X, m, b)
print("====================================================================")
print("                       FINAL CONVERGENCE SUMMARY                    ")
print("====================================================================")
print(f"Final Weights (m)      : {m}")
print(f"Final Bias Vector (b)  : {b}")
print(f"Final Model Predictions: {final_predictions}")
print(f"True Target Vector (y) : {y}")
print("====================================================================\n")

# =====================================================================
# 4. VISUALIZATION MATPLOTLIB SUBPLOTS
# =====================================================================
plt.figure(figsize=(14, 5))

# Plot 1: Parameter changes over iterations
plt.subplot(1, 2, 1)
plt.plot(history_m1, label='m1 (Weight 1)', color='#005088', marker='o', linewidth=2)
plt.plot(history_m2, label='m2 (Weight 2)', color='#11caa0', marker='s', linewidth=2)
plt.plot(history_b, label='b (Bias Value)', color='#d11a2a', marker='^', linewidth=2)
plt.title('Weight & Bias Trajectory Over Iterations', fontsize=12, fontweight='bold')
plt.xlabel('Iteration Index', fontsize=10)
plt.ylabel('Parameter Value', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

# Plot 2: Error Curve (MSE Change)
plt.subplot(1, 2, 2)
plt.plot(history_loss, color='#005088', linewidth=3, label='MSE Cost')
plt.title('Mean Squared Error (Cost) Reduction Curve', fontsize=12, fontweight='bold')
plt.xlabel('Iteration Index', fontsize=10)
plt.ylabel('MSE Value ($J$)', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.show()